This notebook showcases a couple of features that the infrastructure of the library makes available; namely, it will show how instruments can use different so-called pricing engines to calculate their prices (each engine implementing a given model and/or numerical method) and how engines and instruments can be notified of changes in their input data and react accordingly.

<h2>Setup</h2>
To begin, we import the QuantLib module and set up the global evaluation date.

In [1]:
import QuantLib as ql

today = ql.Date(7, ql.March, 2024)
ql.Settings.instance().evaluationDate = today

<h2>The instrument</h2>

In this notebook, we’ll leave fixed-income and take a textbook instrument example: a European option.

Building the option requires only the specification of its contract, so its payoff (it’s a call option with strike at 100) and its exercise, three months from today’s date. The instrument doesn’t take any market data; they will be selected and passed later, depending on the calculation method.

In [2]:
option = ql.EuropeanOption(
    ql.PlainVanillaPayoff(ql.Option.Call, 100.0),
    ql.EuropeanExercise(ql.Date(7, ql.June, 2024))
)

<h2>A first pricing method</h2>

The different pricing methods are implemented as pricing engines holding the required market data. The first we’ll use is the one encapsulating the analytic Black-Scholes formula.

First, we collect the quoted market data. We’ll assume flat risk-free rate and volatility, so they can be expressed by SimpleQuote instances:

In [3]:
u = ql.SimpleQuote(100.0)
r = ql.SimpleQuote(0.01)
sigma = ql.SimpleQuote(0.20)

In order to build the engine, the market data are encapsulated in a Black-Scholes process object. The process can use full-fledged term structures, so it can include time-dependency and smiles. In this case, for simplicity, we build flat curves for the risk-free rate and the volatility.

In [4]:
riskFreeCurve = ql.FlatForward(
    0, ql.TARGET(),ql.QuoteHandle(r), ql.Actual360()
)
volatility = ql.BlackConstantVol(
    0, ql.TARGET(), ql.QuoteHandle(sigma), ql.Actual360()
)

Now we can instantiate the process with the underlying value and the curves we just built. The inputs are all stored into handles, so that we could change the quotes and curves used if we wanted. I’ll skip over this for the time being.

In [5]:
process = ql.BlackScholesProcess(
    ql.QuoteHandle(u),
    ql.YieldTermStructureHandle(riskFreeCurve),
    ql.BlackVolTermStructureHandle(volatility)
)

Once we have the process, we can finally use it to build the engine…

In [6]:
engine = ql.AnalyticEuropeanEngine(process)

…and once we have the engine, we can set it to the option and evaluate the latter.

In [7]:
option.setPricingEngine(engine)
print(option.NPV())

4.155543462156206


Depending on the instrument and the engine, we can also ask for other results; in this case, we can ask for Greeks.

In [8]:
print(option.delta())
print(option.gamma())
print(option.vega())

0.5302223303784392
0.03934493301271913
20.109632428723106


<h2>Market changes</h2>

In [9]:
u.setValue(105.0)
print(option.NPV())

7.27556357927846


Other market data also affect the value, of course.

In [10]:
r.setValue(0.02)
print(option.NPV())

7.448878025811257


In [11]:
sigma.setValue(0.15)
print(option.NPV())

6.596556078273312


<h2>Date changes</h2>

Just as it does when inputs are modified, the value also changes if we advance the evaluation date. Let’s look first at the value of the option when its underlying is worth 105 and there’s still three months to exercise…

In [12]:
u.setValue(105.0)
r.setValue(0.01)
sigma.setValue(0.20)
print(option.NPV())

7.27556357927846


…and then move to a date two months before exercise.

In [13]:
ql.Settings.instance().evaluationDate= ql.Date(7, ql.April, 2024)
print(option.NPV())

6.535204576446796


<h2>A note on the option value on its exercise date</h2>

In the default library configuration, the instrument is considered to have expired when it reaches the exercise date, so its returned value goes down to 0.

In [14]:
ql.Settings.instance().evaluationDate= ql.Date(7, ql.June, 2024)
print(option.NPV())

0.0


It’s possible to tweak the configuration so that the instrument is still considered alive.

In [15]:
ql.Settings.instance().includeReferenceDateEvents = True

The above changes the settings, but doesn’t send a notification to the instrument so we need to trigger an explicit recalculation. Normally, though, one would change the setting at the start of one’s program so this step would be unnecessary.

In [16]:
option.recalculate()
print(option.NPV())

5.0


In [17]:
print(option.delta())
print(option.vega())

1.0
0.0


<h2>Other pricing model</h2>
 I’ll first set the input data back to what they were previously and output the Black-Scholes price.

In [20]:
ql.Settings.instance().evaluationDate = today
u.setValue(105.0)
r.setValue(0.01)
sigma.setValue(0.20)

In [21]:
print(option.NPV())

7.27556357927846


Let’s say that we want to use a Heston model to price the option. What we have to do is to instantiate the corresponding class with the desired inputs (here I’ll skip the calibration and pass precalculated parameters)…

In [22]:
model= ql.HestonModel(
    ql.HestonProcess(
        ql.YieldTermStructureHandle(riskFreeCurve),
        ql.YieldTermStructureHandle(
            ql.FlatForward(0, ql.TARGET(), 0.0, ql.Actual360())
        ),
        ql.QuoteHandle(u),
        0.04,
        0.1,
        0.01,
        0.05,
        -0.75
    )
)

…pass it to the corresponding engine, and set the new engine to the option.

In [23]:
engine = ql.AnalyticHestonEngine(model)
option.setPricingEngine(engine)

In [24]:
print(option.NPV())

7.295356086978634


<h2>Lazy recalculation</h2>

One last thing. Up to now, we haven’t really seen evidence of notifications going around. After all, the instrument might just have recalculated its value every time we asked it, regardless of notifications. What I’m going to show, instead, is that the option doesn’t just recalculate every time anything changes; it also avoids recalculations when nothing has changed.

We’ll switch to a Monte Carlo engine, which takes a few seconds to run the required simulation.

In [28]:
engine = ql.MCEuropeanEngine(
    process, "PseudoRandom", timeSteps=20, requiredSamples=500_000
)
option.setPricingEngine(engine)

When we ask for the option value, we have to wait a noticeable time for the calculation to finish (for those of you reading this in a non-interactive way, I’ll also have the notebook output the time)…

In [29]:
%time print(option.NPV())

7.294517408930112
CPU times: user 2.72 s, sys: 1.58 ms, total: 2.72 s
Wall time: 2.7 s


In [30]:
%time print(option.NPV())

7.294517408930112
CPU times: user 275 μs, sys: 0 ns, total: 275 μs
Wall time: 232 μs


If we change anything (e.g., the underlying value)…

In [31]:
u.setValue(104.0)

…the option is notified of the change, and the next call to NPV will again take a while.

In [32]:
%time print(option.NPV())

6.574127017232327
CPU times: user 2.79 s, sys: 15.3 ms, total: 2.81 s
Wall time: 2.8 s
